In [1]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from pathlib import Path

import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

import torch

import pathlib

/home/andrea/miniforge3/envs/baker/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_lean_files(directory_path):
    print(f"Recursively scanning '{directory_path}' for Lean files...")
    docs = []
    # rglob handles deeply nested directories automatically
    for path in Path(directory_path).rglob("*.lean"):
        try:
            # Explicit utf-8 prevents crashes on math unicode symbols like ∀, ∃, ⊢
            with open(path, "r", encoding="utf-8") as f:
                text = f.read()
                docs.append(Document(page_content=text, metadata={"source": str(path)}))
        except Exception as e:
            print(f"Skipping {path} due to error: {e}")
    print(f"Successfully loaded {len(docs)} Lean documents.")
    return docs

docs = load_lean_files("../mathlib4/Mathlib")

Recursively scanning '../mathlib4/Mathlib' for Lean files...
Successfully loaded 8125 Lean documents.


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        r"\nnamespace\s+",
        r"\nsection\s+",
        r"\ntheorem\s+",
        r"\nlemma\s+",
        r"\ndef\s+",
        r"\ninductive\s+",
        r"\nstructure\s+",
        r"\nexample\s+",
        r"\nopen\s+",      # Sometimes splitting at open statements is clean
        r"\n\n", 
        r"\n", 
        r" "
    ],
    is_separator_regex=True,
    chunk_size=1000,
    chunk_overlap=100
)
chunks = text_splitter.split_documents(docs)

In [4]:
chunks[1052]

Document(metadata={'source': '../mathlib4/Mathlib/Probability/StrongLaw.lean'}, page_content="theorem strong_law_Lp {p : ℝ≥0∞} (hp : 1 ≤ p) (hp' : p ≠ ∞) (X : ℕ → Ω → E)\n    (hℒp : MemLp (X 0) p μ) (hindep : Pairwise ((· ⟂ᵢ[μ] ·) on X))\n    (hident : ∀ i, IdentDistrib (X i) (X 0) μ μ) :\n    Tendsto (fun (n : ℕ) => eLpNorm (fun ω => (n : ℝ)⁻¹ • (∑ i ∈ range n, X i ω) - μ[X 0]) p μ)\n      atTop (𝓝 0) := by\n  -- First exclude the trivial case where the space is not a probability space\n  by_cases h : ∀ᵐ ω ∂μ, X 0 ω = 0\n  · have I : ∀ᵐ ω ∂μ, ∀ i, X i ω = 0 := by\n      rw [ae_all_iff]\n      intro i\n      exact (hident i).symm.ae_snd (p := fun x ↦ x = 0) measurableSet_eq h\n    have A (n : ℕ) : eLpNorm (fun ω => (n : ℝ)⁻¹ • (∑ i ∈ range n, X i ω) - μ[X 0]) p μ = 0 := by\n      simp only [integral_eq_zero_of_ae h, sub_zero]\n      apply eLpNorm_eq_zero_of_ae_zero\n      filter_upwards [I] with ω hω\n      simp [hω]\n    simp [A]\n  -- Then use ae convergence and uniform integrability

In [5]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13362.20it/s]


In [6]:
chunk_texts = [chunk.page_content for chunk in chunks]

if Path('./embedding_matrix.npy').is_file():
    embedding_matrix = np.load('embedding_matrix.npy')
    dimension = embedding_matrix.shape[1]
else:
    raw_embeddings = embeddings.embed_documents(chunk_texts)
    embedding_matrix = np.array(raw_embeddings).astype('float32')
    dimension = embedding_matrix.shape[1]
    np.save('embedding_matrix.npy', embedding_matrix)

In [7]:
model_id = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)


In [8]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype="auto",
    device_map="cuda:0"
)

Loading weights: 100%|██████████| 218/218 [00:00<00:00, 508.30it/s]


In [9]:

pipe = pipeline( 
    "text-generation", 
    model=model, 
    tokenizer=tokenizer, 
    max_new_tokens=300, 
    temperature=0.1
)
llm = HuggingFacePipeline(pipeline=pipe)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [10]:
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embedding_matrix)

In [31]:
def ask_local_rag(query, k=3):
    # Embed user query
    query_vector = np.array([embeddings.embed_query(query)]).astype('float32')
    
    # Search FAISS index natively
    distances, indices = faiss_index.search(query_vector, k)
    
    # Consolidate retrieved contexts
    retrieved_contexts = []
    for idx in indices[0]:
        if idx != -1:
            retrieved_contexts.append(chunk_texts[idx])
            
    context_str = "\n---\n".join(retrieved_contexts)
    
    # Build prompt using modern LangChain invocation syntax
    system_prompt = (
        f"You are a helpful expert assistant specializing in formal verification and the Lean theorem prover.\n"
        f"Use the following chunks of retrieved project code context to answer the user's question accurately.\n"
        f"If the answer cannot be derived from the context, state that you don't have enough context.\n"
        f"Do not add new questions. Only answer below after the \"Answer\" clause. Do not add additional questions and answers.\n\n"
        f"Context:\n{context_str}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    
    return system_prompt, llm.invoke(system_prompt)

In [33]:
ctxt, ans = ask_local_rag(
    "How do you prove the spectral theorem? Provide me with the Lean tactics used in the proof.",
    k=10
)
print("========================== CONTEXT AND PROMPT ==========================")
print(ctxt)
print("========================== ANSWER ==========================")
print(ans.replace(ctxt, ""))

[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


========================== CONTEXT AND PROMPT ==========================
You are a helpful expert assistant specializing in formal verification and the Lean theorem prover.
Use the following chunks of retrieved project code context to answer the user's question accurately.
If the answer cannot be derived from the context, state that you don't have enough context.
Do not add new questions. Only answer below after the "Answer" clause. Do not add additional questions and answers.

Context:
/-
Copyright (c) 2022 Alexander Bentkamp. All rights reserved.
Released under Apache 2.0 license as described in the file LICENSE.
Authors: Alexander Bentkamp
-/
module

public import Mathlib.Algebra.Star.UnitaryStarAlgAut
public import Mathlib.Analysis.InnerProductSpace.Spectrum
public import Mathlib.Analysis.Matrix.Hermitian
public import Mathlib.LinearAlgebra.Eigenspace.Matrix
public import Mathlib.LinearAlgebra.Matrix.Charpoly.Eigs
public import Mathlib.LinearAlgebra.Matrix.Rank

/-! # Spectral theo